In [9]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.nn.functional import pad
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# GPU 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")

# 데이터 경로 설정 및 클래스 목록 자동 생성
data_dir = 'C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/data/camter_tent_data'  # 데이터셋 경로
img_size = (224, 224)

# 클래스 목록 자동 생성 (디렉터리만 포함)
classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
print(f"클래스 목록: {classes}")

def load_data(data_dir, classes):
    images = []
    labels = []
    print("데이터 로딩 시작...")
    for class_index, class_name in enumerate(classes):
        class_path = os.path.join(data_dir, class_name)
        print(f"{class_name} 이미지 로딩 중...")
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            img = plt.imread(img_path)  # 이미지 로드
            img = transforms.ToTensor()(img)  # 이미지를 텐서로 변환
            images.append(img)
            labels.append(class_index)
    print(f"총 {len(images)}개의 이미지 로드 완료")
    return images, labels

# 데이터 증강 설정
transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def custom_collate_fn(batch):
    # 배치 내 최대 크기 확인
    max_height = max([img.size(1) for img, _ in batch])
    max_width = max([img.size(2) for img, _ in batch])

    # 패딩을 추가하여 크기를 맞춤
    padded_images = []
    labels = []
    for img, label in batch:
        _, h, w = img.size()
        padded_img = pad(img, (0, max_width - w, 0, max_height - h))  # (왼쪽, 오른쪽, 위, 아래)
        padded_images.append(padded_img)
        labels.append(label)

    return torch.stack(padded_images), torch.tensor(labels)

def create_model():
    print("모델 생성 중...")
    # EfficientNet B0 모델 로드
    model = models.efficientnet_b0(pretrained=True)

    # 기존 가중치 미세 조정을 위해 일부 레이어 해제
    for param in model.parameters():
        param.requires_grad = False
    for param in model.features[-1].parameters():
        param.requires_grad = True

    # 모델 구조 변경
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
    model = model.to(device)
    print("모델 생성 완료")
    return model

def train_model():
    # 데이터 로드 및 전처리
    images, labels = load_data(data_dir, classes)
    X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=0.2, random_state=42)

    print(f"학습 데이터 크기: {len(X_train)}")
    print(f"검증 데이터 크기: {len(X_val)}")

    # 데이터 로더 생성
    train_dataset = [(img, label) for img, label in zip(X_train, y_train)]
    val_dataset = [(img, label) for img, label in zip(X_val, y_val)]
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=custom_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)

    # 모델 생성
    model = create_model()

    # 손실 함수 및 옵티마이저 설정
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    print("모델 학습 시작...")
    epochs = 3000
    best_val_acc = 0.0

    for epoch in range(epochs):
        # ===== 학습 =====
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

        train_accuracy = correct_train / total_train

        # ===== 검증 =====
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)

        val_accuracy = correct_val / total_val

        # ===== 결과 출력 =====
        print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_accuracy:.4f}, Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_accuracy:.4f}")

        # 최적 모델 저장
        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            torch.save(model.state_dict(), 'best_tent_classifier.pth')
            print("최적 모델 저장")

    print(f"최종 검증 정확도: {best_val_acc:.4f}")
    return model

if __name__ == "__main__":
    print("텐트 상품 분류 모델 학습을 시작합니다...")
    model = train_model()


사용 중인 디바이스: cuda
클래스 목록: ['0 헬리녹스 알파인돔', '1 헬리녹스 노나돔', '10 헬스포츠 김레 패밀리', '2 헬리녹스 브이타프', '3 스노우피크 도크돔', '4 스노우피크 랜드브리즈 Pro.4', '5 스노우피크 랜드브리즈 Pro.3', '6 스노우피크 어메니티돔M', '7 스노우피크 어매니티돔S', '8 코오롱스포츠 에어로라이트', '9 힐레베르그 알락']
텐트 상품 분류 모델 학습을 시작합니다...
데이터 로딩 시작...
0 헬리녹스 알파인돔 이미지 로딩 중...
1 헬리녹스 노나돔 이미지 로딩 중...
10 헬스포츠 김레 패밀리 이미지 로딩 중...
2 헬리녹스 브이타프 이미지 로딩 중...
3 스노우피크 도크돔 이미지 로딩 중...
4 스노우피크 랜드브리즈 Pro.4 이미지 로딩 중...
5 스노우피크 랜드브리즈 Pro.3 이미지 로딩 중...
6 스노우피크 어메니티돔M 이미지 로딩 중...
7 스노우피크 어매니티돔S 이미지 로딩 중...
8 코오롱스포츠 에어로라이트 이미지 로딩 중...
9 힐레베르그 알락 이미지 로딩 중...
총 824개의 이미지 로드 완료
학습 데이터 크기: 659
검증 데이터 크기: 165
모델 생성 중...
모델 생성 완료
모델 학습 시작...
Epoch [1/3000] - Train Loss: 2.3817, Train Acc: 0.1381, Val Loss: 2.3609, Val Acc: 0.1758
최적 모델 저장
Epoch [2/3000] - Train Loss: 2.2956, Train Acc: 0.2094, Val Loss: 2.3018, Val Acc: 0.1758
Epoch [3/3000] - Train Loss: 2.2309, Train Acc: 0.2519, Val Loss: 2.2366, Val Acc: 0.2061
최적 모델 저장
Epoch [4/3000] - Train Loss: 2.1666, Train Acc: 0.2838, Val Loss: 2.1780, Val Acc: 0.